In [1]:
!pip install transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
import torch
import numpy as np
from transformers import AutoModel
from datasets import load_dataset
from typing import Tuple
from itertools import combinations
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import make_scorer

In [3]:
model = AutoModel.from_pretrained("Synthyra/ESMplusplus_small", trust_remote_code=True, torch_dtype=torch.float16).cuda()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

modeling_esm_plusplus.py:   0%|          | 0.00/40.7k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_small:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [4]:
data = load_dataset('GleghornLab/optimal_temperature')
all_seqs = data['train']['seqs'] + data['valid']['seqs'] + data['test']['seqs']
len(all_seqs)

README.md:   0%|          | 0.00/527 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/714k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/95.3k [00:00<?, ?B/s]

valid-00000-of-00001.parquet:   0%|          | 0.00/96.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1516 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/190 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/190 [00:00<?, ? examples/s]

1896

In [5]:
emb_dict = model.embed_dataset(
    sequences=all_seqs, # list of protein strings
    batch_size=8, # embedding batch size
    max_len=2048, # truncate to max_len
    full_embeddings=True, # return residue-wise embeddings
    full_precision=False, # store as float32
    num_workers=0, # data loading num workers
    sql=False, # return dictionary of sequences and embeddings
)

Embedding batches:   0%|          | 0/236 [00:00<?, ?it/s]

In [6]:
class Pooler:
    def __init__(self, hf_dataset, emb_dict):
        self.train_seqs, self.train_labels = hf_dataset['train']['seqs'], hf_dataset['train']['labels']
        self.valid_seqs, self.valid_labels = hf_dataset['valid']['seqs'], hf_dataset['valid']['labels']
        self.test_seqs, self.test_labels = hf_dataset['test']['seqs'], hf_dataset['test']['labels']
        self.emb_dict = emb_dict

        self.pooling_options = {
            'mean': self.mean_pooling,
            'max': self.max_pooling,
            'min': self.min_pooling,
            'norm': self.norm_pooling,
            'prod': self.prod_pooling,
            'median': self.median_pooling,
            'std': self.std_pooling,
            'var': self.var_pooling,
            'cls': self.cls_pooling,
            'eos': self.eos_pooling
        }

    def mean_pooling(self, emb): # (L, d) -> (d)
        return emb.mean(dim=0)

    def max_pooling(self, emb): # (L, d) -> (d)
        return emb.max(dim=0).values

    def min_pooling(self, emb): # (L, d) -> (d)
        return emb.min(dim=0).values

    def norm_pooling(self, emb): # (L, d) -> (d)
        return emb.norm(dim=0, p=2)

    def prod_pooling(self, emb): # (L, d) -> (d)
        length = emb.shape[0]
        return emb.prod(dim=0) / length

    def median_pooling(self, emb): # (L, d) -> (d)
        return emb.median(dim=0).values

    def std_pooling(self, emb): # (L, d) -> (d)
        return emb.std(dim=0)

    def var_pooling(self, emb): # (L, d) -> (d)
        return emb.var(dim=0)

    def cls_pooling(self, emb): # (L, d) -> (d)
        return emb[0, :]

    def eos_pooling(self, emb): # (L, d) -> (d)
        return emb[-1, :]

    def get_pooled_embeddings(self, pooling_types: Tuple[str, str]): # [mean, max]
        """
        pooler = Pooler(hf_dataset, emb_dict)
        pooling_types = ['mean', 'max']
        X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)
        # X (n_samples, n_features)
        # y (n_samples)
        """
        for pool_type in pooling_types:
            if pool_type not in self.pooling_options:
                raise ValueError(f"Invalid pooling type: {pool_type}")

        train_embeddings, valid_embeddings, test_embeddings = [], [], []
        train_labels = np.array(self.train_labels)
        valid_labels = np.array(self.valid_labels)
        test_labels = np.array(self.test_labels)

        # concatenate all pooling for each sequence
        for seq in self.train_seqs:
            final_emb = []
            emb = self.emb_dict[seq] # (L, d)
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb) # (d)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten() # (d * len(pooling_type))
            train_embeddings.append(final_emb)

        train_embeddings = torch.stack(train_embeddings).numpy().astype(np.float32)

        for seq in self.valid_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            valid_embeddings.append(final_emb)

        valid_embeddings = torch.stack(valid_embeddings).numpy().astype(np.float32)

        for seq in self.test_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            test_embeddings.append(final_emb)

        test_embeddings = torch.stack(test_embeddings).numpy().astype(np.float32)

        return train_embeddings, train_labels, valid_embeddings, valid_labels, test_embeddings, test_labels


In [50]:
pooling_options = ['mean', 'max', 'min', 'norm', 'prod', 'median', 'std', 'var', 'cls', 'eos']

experiments = list(combinations(pooling_options, 2)) # for the first 45, number of features 960 * 2
for pooling_option in pooling_options:
    experiments.append((pooling_option,)) # for the last 10, number of features 960

print(experiments)
print(len(experiments))

[('mean', 'max'), ('mean', 'min'), ('mean', 'norm'), ('mean', 'prod'), ('mean', 'median'), ('mean', 'std'), ('mean', 'var'), ('mean', 'cls'), ('mean', 'eos'), ('max', 'min'), ('max', 'norm'), ('max', 'prod'), ('max', 'median'), ('max', 'std'), ('max', 'var'), ('max', 'cls'), ('max', 'eos'), ('min', 'norm'), ('min', 'prod'), ('min', 'median'), ('min', 'std'), ('min', 'var'), ('min', 'cls'), ('min', 'eos'), ('norm', 'prod'), ('norm', 'median'), ('norm', 'std'), ('norm', 'var'), ('norm', 'cls'), ('norm', 'eos'), ('prod', 'median'), ('prod', 'std'), ('prod', 'var'), ('prod', 'cls'), ('prod', 'eos'), ('median', 'std'), ('median', 'var'), ('median', 'cls'), ('median', 'eos'), ('std', 'var'), ('std', 'cls'), ('std', 'eos'), ('var', 'cls'), ('var', 'eos'), ('cls', 'eos'), ('mean',), ('max',), ('min',), ('norm',), ('prod',), ('median',), ('std',), ('var',), ('cls',), ('eos',)]
55


In [8]:
def pearson_metric(y_true, y_pred):
    return pearsonr(y_true, y_pred).correlation

# Add a __name__ attribute to the function
pearson_metric.__name__ = "pearson_metric"
pearson_scorer = make_scorer(pearson_metric, greater_is_better=True)

In [55]:
pooler = Pooler(data, emb_dict)

results = {}
for exp in pooling_options:
  try:
      X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings([exp])
      print(f"Successfully pooled with {exp}")
  except ValueError as e:
      print(e)
  test_model = HistGradientBoostingRegressor(early_stopping=False, l2_regularization=1.0,
                              learning_rate=0.01, loss='absolute_error',
                              max_depth=10, max_iter=500, min_samples_leaf=5,
                              scoring='neg_mean_squared_error')
  test_model.fit(X_train, y_train)
  score = pearson_scorer(test_model, X_test, y_test)
  print(f"Experiment: {exp}")
  print(f"Pearson Score Tested: {score}")
  results[exp] = score


Successfully pooled with mean


AttributeError: 'ESMplusplusModel' object has no attribute 'score'

In [52]:
print(results)

{'mean': 0.768678902059134, 'max': 0.7178651661618501, 'min': 0.7243617019027416, 'norm': 0.7416761085317732, 'prod': nan, 'median': 0.7498929545562119, 'std': 0.6221274047645676, 'var': 0.6227210062744706, 'cls': 0.7791859047063732, 'eos': 0.706358201441954}


In [41]:
print(X_train[1500][2129])
print(np.where(X_train[1500] == X_train[1500].min()))

-0.90771484
(array([2129]),)
